
# Extraction des phrases où un personnage apparaît avec un mot cible

Ce notebook :

1. lit le fichier `.ods` des personnages ;
2. construit, pour chaque personnage, une liste de variantes de noms à partir des **3 premières colonnes** :
   - `NOMS HUMAINS`
   - `SECOND NOM`
   - `TROISIEME NOM`
3. prend en compte les combinaisons demandées :
   - `1 + 2`
   - `2 + 3`
   - `1 + 2 + 3`
   - `1 + 3`
4. repère dans le CSV les **phrases** qui contiennent au moins un mot cible (`mort`, `décéder`, `mourir`) ;
5. vérifie quels personnages apparaissent dans ces phrases ;
6. génère un tableau de sortie avec :
   - l'**identifiant unique** du personnage ;
   - tous ses **alias** ;
   - la **phrase** concernée.

## Logique de détection

- Les **mots cibles** sont recherchés dans la colonne `text_lemma` par défaut, pour repérer plus facilement les variantes fléchies.
- La **phrase exportée** provient de la colonne `text`.
- Les **noms de personnages** sont recherchés dans la phrase `text`.
- Une ligne = **une phrase pour un personnage donné**.
  Si deux personnages apparaissent dans la même phrase avec un mot cible, la phrase apparaîtra sur deux lignes.


In [1]:

# Paramètres
ODS_PATH = "Noms personnages de Pern Noms de famille.ods"
CSV_PATH = "La-ballade-de-Pern-intégrale-sentences-lemma.csv"
ODS_SHEET = "Nom et genre persos"

# True = compte aussi les noms simples (recommandé)
# False = ne compte QUE 1+2, 2+3, 1+2+3, 1+3
INCLUDE_SINGLE_NAMES = True

# Colonnes du CSV
TEXT_COLUMN = "text"           # phrase à exporter
LEMMA_COLUMN = "text_lemma"    # colonne utilisée pour repérer les mots cibles

# Mots cibles
TARGET_WORDS = ["décéder", "mourir", "disparaître", "disparu", "tuer", "assassiner", "deuil", "endeuiller", "funèbre", "enterrement", "défunt", "dépouille", "n'est plus"]

# Fichiers de sortie
OUTPUT_XLSX = "pern_personnages_mort_phrases.xlsx"
OUTPUT_CSV = "pern_personnages_mort_phrases.csv"
OUTPUT_ALIASES_CSV = "pern_aliases_utilises.csv"


In [2]:

import pandas as pd
import re
import unicodedata
from pathlib import Path


In [3]:

# Lecture des fichiers
names_df = pd.read_excel(ODS_PATH, sheet_name=ODS_SHEET, engine="odf")
text_df = pd.read_csv(CSV_PATH)


C:\Users\marct\AppData\Local\Temp\ipykernel_15880\1861214423.py:3: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  text_df = pd.read_csv(CSV_PATH)


In [4]:

# Vérifications minimales
required_name_cols = ["NOMS HUMAINS", "SECOND NOM", "TROISIEME NOM", "Id unique"]
required_csv_cols = [TEXT_COLUMN, LEMMA_COLUMN]

missing_ods = [c for c in required_name_cols if c not in names_df.columns]
missing_csv = [c for c in required_csv_cols if c not in text_df.columns]

if missing_ods:
    raise ValueError(f"Colonnes manquantes dans le ODS : {missing_ods}")

if missing_csv:
    raise ValueError(f"Colonnes manquantes dans le CSV : {missing_csv}")

print("Vérification OK.")


Vérification OK.


In [5]:

def clean_part(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    return x if x else None


def normalize_text(s):
    """Minuscule + suppression des accents + espaces normalisés."""
    if pd.isna(s):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s


def build_aliases(row, include_single_names=True):
    """Construit les variantes d'un personnage à partir des 3 premières colonnes."""
    p1 = clean_part(row["NOMS HUMAINS"])
    p2 = clean_part(row["SECOND NOM"])
    p3 = clean_part(row["TROISIEME NOM"])

    aliases = []

    # Formes simples
    if include_single_names:
        for p in (p1, p2, p3):
            if p:
                aliases.append(p)

    # Combinaisons demandées
    if p1 and p2:
        aliases.append(f"{p1} {p2}")      # 1 + 2
    if p2 and p3:
        aliases.append(f"{p2} {p3}")      # 2 + 3
    if p1 and p2 and p3:
        aliases.append(f"{p1} {p2} {p3}") # 1 + 2 + 3
    if p1 and p3:
        aliases.append(f"{p1} {p3}")      # 1 + 3

    # Déduplication en conservant l'ordre
    seen = set()
    clean_aliases = []
    for a in aliases:
        a = re.sub(r"\s+", " ", a).strip()
        key = normalize_text(a)
        if a and key not in seen:
            seen.add(key)
            clean_aliases.append(a)

    # Les alias longs d'abord pour mieux repérer les formes composées
    clean_aliases = sorted(clean_aliases, key=lambda x: (-len(x), normalize_text(x)))
    return clean_aliases


names_df = names_df.copy()
names_df["aliases"] = names_df.apply(
    lambda row: build_aliases(row, include_single_names=INCLUDE_SINGLE_NAMES),
    axis=1
)

names_df[["NOMS HUMAINS", "SECOND NOM", "TROISIEME NOM", "Id unique", "aliases"]].head(15)


,NOMS HUMAINS,SECOND NOM,TROISIEME NOM,Id unique,aliases
0,A. C.,Sopers,NaN,940.0,"[A. C. Sopers, Sopers, A. C.]"
1,A’dan,NaN,NaN,270.0,[A’dan]
2,A’murry,NaN,NaN,271.0,[A’murry]
3,Abula,NaN,NaN,2.0,[Abula]
4,Abuna,NaN,NaN,3.0,[Abuna]
5,Adessa,NaN,NaN,4.0,[Adessa]
6,Adrea,Beljeth,NaN,5.0,"[Adrea Beljeth, Beljeth, Adrea]"
7,Afnor,NaN,NaN,272.0,[Afnor]
8,Aida,NaN,NaN,6.0,[Aida]
9,Aisling,Hempenstahl,NaN,941.0,"[Aisling Hempenstahl, Hempenstahl, Aisling]"


In [6]:

# Tableau d'audit des alias réellement utilisés
aliases_audit = names_df[["Id unique", "NOMS HUMAINS", "SECOND NOM", "TROISIEME NOM", "aliases"]].copy()
aliases_audit["aliases"] = aliases_audit["aliases"].apply(lambda x: " | ".join(x))
aliases_audit.to_csv(OUTPUT_ALIASES_CSV, index=False)
print(f"Fichier alias exporté : {OUTPUT_ALIASES_CSV}")
aliases_audit.head(10)


Fichier alias exporté : pern_aliases_utilises.csv


,Id unique,NOMS HUMAINS,SECOND NOM,TROISIEME NOM,aliases
0,940.0,A. C.,Sopers,NaN,A. C. Sopers | Sopers | A. C.
1,270.0,A’dan,NaN,NaN,A’dan
2,271.0,A’murry,NaN,NaN,A’murry
3,2.0,Abula,NaN,NaN,Abula
4,3.0,Abuna,NaN,NaN,Abuna
5,4.0,Adessa,NaN,NaN,Adessa
6,5.0,Adrea,Beljeth,NaN,Adrea Beljeth | Beljeth | Adrea
7,272.0,Afnor,NaN,NaN,Afnor
8,6.0,Aida,NaN,NaN,Aida
9,941.0,Aisling,Hempenstahl,NaN,Aisling Hempenstahl | Hempenstahl | Aisling


In [7]:

def compile_character_pattern(aliases):
    """Crée une regex qui repère les alias du personnage avec bornes souples."""
    if not aliases:
        return None

    escaped = [re.escape(normalize_text(a)) for a in aliases if a]
    if not escaped:
        return None

    pattern = r"(?<!\w)(?:" + "|".join(escaped) + r")(?!\w)"
    return re.compile(pattern, flags=re.IGNORECASE)


def compile_target_pattern(target_words):
    normalized_targets = [re.escape(normalize_text(w)) for w in target_words if w]
    pattern = r"(?<!\w)(?:" + "|".join(normalized_targets) + r")(?!\w)"
    return re.compile(pattern, flags=re.IGNORECASE)


def find_all_matches(pattern, text):
    if pattern is None:
        return []
    return pattern.findall(text)


In [8]:

# Préparation du CSV
text_df = text_df.copy()
text_df[TEXT_COLUMN] = text_df[TEXT_COLUMN].fillna("").astype(str)
text_df[LEMMA_COLUMN] = text_df[LEMMA_COLUMN].fillna("").astype(str)

text_df["text_norm"] = text_df[TEXT_COLUMN].apply(normalize_text)
text_df["lemma_norm"] = text_df[LEMMA_COLUMN].apply(normalize_text)

target_pattern = compile_target_pattern(TARGET_WORDS)
text_df["target_matches"] = text_df["lemma_norm"].apply(lambda x: sorted(set(find_all_matches(target_pattern, x))))
text_df["has_target_word"] = text_df["target_matches"].apply(bool)

candidate_df = text_df[text_df["has_target_word"]].copy()

print(f"Nombre total de phrases : {len(text_df):,}")
print(f"Nombre de phrases contenant un mot cible : {len(candidate_df):,}")

candidate_df[["sentence_id", TEXT_COLUMN, LEMMA_COLUMN, "target_matches"]].head(10)


Nombre total de phrases : 132,850
Nombre de phrases contenant un mot cible : 1,014


,sentence_id,text,text_lemma,target_matches
184,185,Il se permit un petit sourire pour la fantaisi...,Il se permettre un petit sourire pour le fanta...,[disparaitre]
811,812,"Gracieuses comme des flèches, elles amorcèrent...","Gracieuses comme un flèche, lui amorcèrent son...",[disparaitre]
925,926,"La navette atterrirait sans problème, mais l’a...","Le navette atterrir sans problème, mais l’amir...",[mourir]
965,966,"Alarmé, Kenjo se retourna, juste à temps pour ...","Alarmé, Kenjo se retourner, juste à temps pour...",[disparaitre]
1093,1094,"Pour elle, ce n’était qu’une façon de tuer le ...","Pour lui, ce n’être qu’un façon de tuer le temps.",[tuer]
1610,1611,— Je veux pas la tuer !,— Je vouloir pas le tuer !,[tuer]
1613,1614,Je veux rien tuer.,Je vouloir rien tuer.,[tuer]
1719,1720,"Bientôt, il n’y aurait plus aucun endroit sur ...","Bientôt, il n’y avoir plus aucun endroit sur l...",[disparaitre]
1896,1897,"— Il meurt de faim, Sean, dit Sorka, cherchant...","— Il mourir de faim, Sean, dire Sorka, cherche...",[mourir]
1902,1903,Il atterrit juste devant le nouveau-né vacilla...,Il atterrir juste devant le nouveau-naître vac...,[disparaitre]


In [9]:

# Extraction des phrases par personnage
results = []

for i, row in names_df.iterrows():
    char_id = row["Id unique"]
    aliases = row["aliases"]
    aliases_str = " | ".join(aliases)
    char_pattern = compile_character_pattern(aliases)

    if char_pattern is None:
        continue

    char_matches_mask = candidate_df["text_norm"].str.contains(char_pattern, na=False)
    matched_rows = candidate_df[char_matches_mask]

    for _, sent_row in matched_rows.iterrows():
        matched_aliases = sorted(set(find_all_matches(char_pattern, sent_row["text_norm"])))
        results.append({
            "Id unique": char_id,
            "aliases": aliases_str,
            "phrase": sent_row[TEXT_COLUMN],
            "sentence_id": sent_row.get("sentence_id", pd.NA),
            "book": sent_row.get("book", pd.NA),
            "mots_cibles": " | ".join(sent_row["target_matches"]),
            "alias_trouves_dans_phrase": " | ".join(matched_aliases),
        })

    if (i + 1) % 100 == 0:
        print(f"{i + 1} personnages traités / {len(names_df)}")

result_df = pd.DataFrame(results)
print("Extraction terminée.")
print("Nombre de lignes extraites :", len(result_df))
result_df.head(10)


100 personnages traités / 1069
200 personnages traités / 1069
300 personnages traités / 1069
400 personnages traités / 1069
500 personnages traités / 1069
600 personnages traités / 1069
700 personnages traités / 1069
800 personnages traités / 1069
900 personnages traités / 1069
1000 personnages traités / 1069
Extraction terminée.
Nombre de lignes extraites : 769


,Id unique,aliases,phrase,sentence_id,book,mots_cibles,alias_trouves_dans_phrase
0,4.0,Adessa,Ne me dis pas que cette chère Dame Adessa est ...,60657,7.0,mourir,adessa
1,274.0,Alemi,La côte avait disparu dans la soudaine obscuri...,11117,2.0,disparaitre,alemi
2,274.0,Alemi,"Tous les six repartirent vers le large, sautan...",13333,2.0,disparaitre,alemi
3,274.0,Alemi,— Je ne connaissais presque rien de la vie des...,18631,2.0,disparaitre,alemi
4,275.0,Alemi Onclemi Lemi | Alemi Onclemi | Onclemi L...,La côte avait disparu dans la soudaine obscuri...,11117,2.0,disparaitre,alemi
5,275.0,Alemi Onclemi Lemi | Alemi Onclemi | Onclemi L...,"Tous les six repartirent vers le large, sautan...",13333,2.0,disparaitre,alemi
6,275.0,Alemi Onclemi Lemi | Alemi Onclemi | Onclemi L...,— Je ne connaissais presque rien de la vie des...,18631,2.0,disparaitre,alemi
7,276.0,Alessan,"Puis, ce fut le phénomène qu’Alessan attendait...",29904,4.0,disparaitre,alessan
8,276.0,Alessan,– et ne pouvait pas être dans la terre du cham...,34474,4.0,mourir,alessan
9,276.0,Alessan,Alessan les regarda s’éloigner jusqu’à ce que ...,35009,4.0,disparaitre,alessan


In [10]:

# Nettoyage final : une ligne = une phrase pour un personnage
if not result_df.empty:
    result_df = result_df.drop_duplicates(subset=["Id unique", "phrase"]).copy()

    # Tri pour lecture plus facile
    sort_cols = [c for c in ["Id unique", "book", "sentence_id"] if c in result_df.columns]
    result_df = result_df.sort_values(sort_cols).reset_index(drop=True)

print(result_df.shape)
result_df.head(20)


(552, 7)


,Id unique,aliases,phrase,sentence_id,book,mots_cibles,alias_trouves_dans_phrase
0,1.0,Kenjo Fusaiyuki | Fusaiyuki | Kenjo,"Alarmé, Kenjo se retourna, juste à temps pour ...",966,1.0,disparaitre,kenjo
1,1.0,Kenjo Fusaiyuki | Fusaiyuki | Kenjo,Ongola et Kenjo disparurent derrière un tas de...,6009,1.0,disparaitre,kenjo
2,1.0,Kenjo Fusaiyuki | Fusaiyuki | Kenjo,— Comment Kenjo a-t-il été tué ?,6030,1.0,tuer,kenjo
3,1.0,Kenjo Fusaiyuki | Fusaiyuki | Kenjo,Avril dit qu’elle a tué Ongola et Kenjo.,6439,1.0,tuer,kenjo
4,1.0,Kenjo Fusaiyuki | Fusaiyuki | Kenjo,Ou qu’Ongola a tué Kenjo pour les empêcher de ...,6654,1.0,tuer,kenjo
5,1.0,Kenjo Fusaiyuki | Fusaiyuki | Kenjo,Aucun doute que le défunt Kenjo n’ait utilisé ...,127919,15.0,defunt,kenjo
6,4.0,Adessa,Ne me dis pas que cette chère Dame Adessa est ...,60657,7.0,mourir,adessa
7,14.0,Anella,La courtoisie que nous avait inculquée notre d...,38167,5.0,defunt,anella
8,14.0,Anella,"Anella, vêtue d’une lourde robe de brocart, to...",38191,5.0,deuil,anella
9,15.0,Aramina Mina Ara | Aramina Mina | Aramina Ara ...,"sans savoir pourquoi, il était sûr que Thella ...",44875,6.0,tuer,aramina


In [11]:

# Export
#result_df.to_excel(OUTPUT_XLSX, index=False)
result_df.to_csv(OUTPUT_CSV, index=False)

#print(f"Fichier Excel exporté : {OUTPUT_XLSX}")
print(f"Fichier CSV exporté   : {OUTPUT_CSV}")


Fichier CSV exporté   : pern_personnages_mort_phrases.csv
